# Soro-Kev UPP: P0 v2 architecture grid (2x T4)

Decides the lead design for the paper (go/no-go 22 Nov): does the Matryoshka gist pointer (KM) stay within
~2 points of the full-text block pointer (F) at 16 tools, and how does it do at 64 tools where full-text
layouts cannot run? Pass 1 (9 Nov) already has the query-first heads A/B/G/I; this run adds the cached designs.

**Kaggle settings:** GPU T4 x2, Internet on. ~4 h. Gemma 4 runs live in `kaggle_omni_slice.ipynb`.

| ID | Variant |
| :-- | :-- |
| A | Kev pointer (query first, dot) |
| F | block layout, full option text (ReCache-style uncompressed per-tool cache) |
| K / K8 | block + 4 / 8 gists |
| Ksd | block + 4 gists + gist self-distillation |
| KM | block + 8 Matryoshka gists (budgets 1/2/4/8) + distillation: one cache, two-level scoring |

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/nisaral/Soro-Boro.git"
BRANCH = "upp-p0"          # branch with veyra/python/src/soro/routing/upp.py
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
REPO = f"{WORK}/Soro-Boro"
# Option B: upload the repo as a Kaggle dataset and set LOCAL_COPY to its path.
LOCAL_COPY = None  # e.g. "/kaggle/input/soro-boro/Soro-Boro"
if LOCAL_COPY:
    subprocess.run(["cp", "-r", LOCAL_COPY, REPO], check=True)
elif not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install -U "transformers>=5.0" peft bitsandbytes accelerate huggingface_hub safetensors datasets sentence-transformers
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.device_count(), "GPU(s)")

## 0. Architecture bench (inference only): exactness of block / gist / composed caches and latency vs menu size

In [ ]:
!python veyra/training/arch_bench.py --base-model Qwen/Qwen3-0.6B-Base --sizes 6,16,64,256,1024 --out runs/arch_bench_qwen3_0.6b.json

## 1. Parent check
Block isolation exactness, cache exactness, early exits, VRAM and latency for each candidate.
Qwen3.5 is left out on purpose: its Gated DeltaNet layers cannot run the block layout.

In [ ]:
!python veyra/training/parent_check.py --models Qwen/Qwen3-0.6B-Base,Qwen/Qwen3-1.7B-Base --out runs/parent_check.json

## 2. P0 grid on Qwen3-0.6B-Base (both GPUs in parallel)
Same data, seed and steps for every variant. Held-out: train packages carved out for selection,
and a stride sample of `val_zeroshot.jsonl`. Data comes from the public HF dataset.

In [ ]:
COMMON = ("--base-model Qwen/Qwen3-0.6B-Base --train-rows 4800 --eval-rows 400 --batch 8 --max-seq-len 832 --log-every 50 "
          "--toolret 400 --toolret-k 16 --toolret-k-large 64 --toolret-large-n 300")
gpu0 = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 python veyra/training/p0_upp_ablation.py --configs KM,F,K {COMMON} --out runs/p0_gpu0 > runs_p0_gpu0.log 2>&1", shell=True)
gpu1 = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 python veyra/training/p0_upp_ablation.py --configs Ksd,A,K8 {COMMON} --out runs/p0_gpu1 > runs_p0_gpu1.log 2>&1", shell=True)
print("started; tail the logs in the next cell")

In [ ]:
!tail -n 3 runs_p0_gpu0.log; echo ---; tail -n 3 runs_p0_gpu1.log

In [ ]:
gpu0.wait(); gpu1.wait()
import json
res = {}
for d in ("runs/p0_gpu0", "runs/p0_gpu1"):
    res.update(json.load(open(f"{d}/results.json")))
json.dump(res, open("runs/p0_qwen3_0.6b.json", "w"), indent=2)
sys.path.insert(0, "veyra/training")
from p0_upp_ablation import markdown
md_table = markdown(dict(sorted(res.items())))
open("runs/p0_qwen3_0.6b.md", "w").write(md_table)
print(md_table)

## 3. Gemma 4 E2B
Moved to `kaggle_omni_slice.ipynb` (text, speech and screenshot requests on one text cache).

In [ ]:
print("Gemma runs: see kaggle_omni_slice.ipynb")

In [ ]:
!zip -qr /kaggle/working/p0_results.zip runs runs_p0_gpu0.log runs_p0_gpu1.log 2>/dev/null || zip -qr p0_results.zip runs
print("p0_results.zip written")